# DeliverEase – Food Delivery Time Prediction

### Project Overview
DeliverEase is a supervised machine learning regression project that predicts food delivery time (in minutes) using order details, delivery partner information, distance, weather, traffic, and other factors.

### Project Workflow
1. **Data Loading & Inspection** – Understand the dataset and its features.
2. **Data Cleaning** – Handle missing values and inconsistent data.
3. **EDA** – Explore patterns and relationships between features.
4. **Feature Engineering** – Create useful features such as distance and delivery hour.
5. **Preprocessing** – Encode categorical variables and scale numerical features where required.
6. **Model Training** – Train Linear Regression, Decision Tree, Random Forest, and XGBoost.
7. **Evaluation & Optimization** – Compare models, tune hyperparameters, and select the best model.
8. **Model Saving** – Save the final model for future predictions.

**Target Variable:** `Time_taken(min)`  
**Problem Type:** Supervised Learning – Regression

In [1]:
#import libraries
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

In [2]:
df = pd.read_csv('../data/Food_delivery.csv')

In [3]:
df.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


In [4]:
# Rows x columns
df.shape

(45593, 20)

In [5]:
df.info() #data types of the columns and null values

<class 'pandas.DataFrame'>
RangeIndex: 45593 entries, 0 to 45592
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           45593 non-null  str    
 1   Delivery_person_ID           45593 non-null  str    
 2   Delivery_person_Age          45593 non-null  str    
 3   Delivery_person_Ratings      45593 non-null  str    
 4   Restaurant_latitude          45593 non-null  float64
 5   Restaurant_longitude         45593 non-null  float64
 6   Delivery_location_latitude   45593 non-null  float64
 7   Delivery_location_longitude  45593 non-null  float64
 8   Order_Date                   45593 non-null  str    
 9   Time_Orderd                  45593 non-null  str    
 10  Time_Order_picked            45593 non-null  str    
 11  Weatherconditions            45593 non-null  str    
 12  Road_traffic_density         45593 non-null  str    
 13  Vehicle_condition          

## Basic Data Cleaning 

1\. Fix Weatherconditions formatting: Remove the `"conditions "` prefix from the `Weatherconditions` column (e.g., `"conditions Sunny"` → `"Sunny"`).

2\. Remove unwanted whitespace: Strip leading and trailing spaces from all object/string columns using `.str.strip()`.

3\. Handle missing values: Replace literal `'NaN'` strings (including whitespace-padded values) with actual null values (`np.nan`).

4\. Convert data types:

- `Delivery_person_Age` → Numeric
- `Delivery_person_Ratings` → Numeric
- `multiple_deliveries` → Numeric
- `Time_taken(min)` → Integer
- `Order_Date` → Datetime

5\. Handle invalid values:

- Age: 38 records have value `15` (inspect before correction).
- Ratings: 53 records have value `6` (outside the expected 1–5 range).

6\. Check duplicates: Check duplicate rows and duplicate IDs in the `ID` column.

7\. Validate categorical values: Check unique values and handle inconsistent categories in `Weatherconditions`, `Road_traffic_density`, `Festival`, `City`, and other categorical columns.

8\. Validate geographical coordinates:

- Identify 3,640 records with restaurant coordinates `(0, 0)`.
- Inspect delivery coordinates for invalid values.

9\. Detect outliers: Inspect extreme values in numerical columns, especially delivery distance (431 records have calculated distances above 100 km before coordinate corrections).

10\. Clean target variable: Extract numerical values from `Time_taken(min)` (e.g., `"(min) 24"` → `24`). The target ranges from 10 to 54 minutes.

11\. Handle missing values: Impute numerical columns using the median and categorical columns using the mode. Fit imputers only on training data after the train-test split.

12\. Final validation: Recheck dataset shape, missing values, data types, duplicates, and invalid values.

#1 Fix text formatting: Remove the "conditions " prefix from Weatherconditions (e.g., "conditions Sunny" → "Sunny").

In [7]:
df['Weatherconditions']

0             conditions Sunny
1            conditions Stormy
2        conditions Sandstorms
3             conditions Sunny
4            conditions Cloudy
                 ...          
45588         conditions Windy
45589         conditions Windy
45590        conditions Cloudy
45591        conditions Cloudy
45592           conditions Fog
Name: Weatherconditions, Length: 45593, dtype: str

In [8]:
# Note the messy formatting: categories arrive as "conditions Sunny", plus a literal "conditions NaN"
df['Weatherconditions'].value_counts()

Weatherconditions
conditions Fog           7654
conditions Stormy        7586
conditions Cloudy        7536
conditions Sandstorms    7495
conditions Windy         7422
conditions Sunny         7284
conditions NaN            616
Name: count, dtype: int64

In [10]:
# Those "conditions NaN" rows are missing weather stored as text, not real nulls yet
df[df['Weatherconditions']=='conditions NaN']

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
74,0xa9f,BANGRES13DEL01,NaN,NaN,12.935662,77.614130,12.975662,77.654130,11-03-2022,NaN,15:30:00,conditions NaN,NaN,1,Drinks,scooter,0,No,Metropolitian,(min) 19
120,0x6e2,VADRES06DEL01,NaN,NaN,22.312790,73.170283,22.422790,73.280283,02-04-2022,NaN,18:25:00,conditions NaN,NaN,3,Snack,electric_scooter,0,No,Metropolitian,(min) 25
250,0x1b50,PUNERES16DEL01,NaN,NaN,18.536718,73.830327,18.646718,73.940327,08-03-2022,NaN,21:25:00,conditions NaN,NaN,3,Buffet,motorcycle,1,Yes,Urban,(min) 43
348,0x4c1,BANGRES13DEL02,NaN,NaN,12.935662,77.614130,13.065662,77.744130,02-03-2022,NaN,00:05:00,conditions NaN,NaN,3,Drinks,scooter,1,No,Metropolitian,(min) 26
425,0xc003,DEHRES13DEL02,NaN,NaN,-30.366322,-78.070453,30.496322,78.200453,18-02-2022,NaN,22:30:00,conditions NaN,NaN,3,Snack,scooter,1,No,Metropolitian,(min) 20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45332,0x2950,VADRES04DEL02,50,6,0.000000,0.000000,0.020000,0.020000,19-03-2022,NaN,10:40:00,conditions NaN,NaN,3,Drinks,electric_scooter,1,No,Urban,(min) 28
45394,0xdc4,RANCHIRES02DEL03,NaN,NaN,0.000000,0.000000,0.030000,0.030000,05-03-2022,NaN,17:25:00,conditions NaN,NaN,3,Buffet,bicycle,1,No,Metropolitian,(min) 21
45504,0x461,BANGRES07DEL02,NaN,NaN,-12.978453,-77.643685,12.998453,77.663685,17-03-2022,NaN,09:30:00,conditions NaN,NaN,3,Meal,scooter,1,No,NaN,(min) 24
45518,0x863,INDORES05DEL01,NaN,NaN,22.727021,75.884167,22.737021,75.894167,19-03-2022,NaN,11:40:00,conditions NaN,NaN,3,Snack,electric_scooter,0,No,Metropolitian,(min) 25


In [15]:
df['Weatherconditions'] = df['Weatherconditions'].str.replace(
    'conditions ', '', regex=False
)

In [16]:
df['Weatherconditions']

0             Sunny
1            Stormy
2        Sandstorms
3             Sunny
4            Cloudy
            ...    
45588         Windy
45589         Windy
45590        Cloudy
45591        Cloudy
45592           Fog
Name: Weatherconditions, Length: 45593, dtype: str

In [21]:
df['Weatherconditions'].value_counts()

Weatherconditions
Fog           7654
Stormy        7586
Cloudy        7536
Sandstorms    7495
Windy         7422
Sunny         7284
NaN            616
Name: count, dtype: int64

In [22]:
# Step 2: Remove leading and trailing spaces
# Step 3: Replace string 'NaN' with actual missing values

for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].str.strip()
    df[col] = df[col].replace('NaN', np.nan)

In [ ]:
print("Missing values in each column:") #verify for missing values
print(df.isnull().sum())

Missing values in each column:
ID                                0
Delivery_person_ID                0
Delivery_person_Age            1854
Delivery_person_Ratings        1908
Restaurant_latitude               0
Restaurant_longitude              0
Delivery_location_latitude        0
Delivery_location_longitude       0
Order_Date                        0
Time_Orderd                    1731
Time_Order_picked                 0
Weatherconditions               616
Road_traffic_density            601
Vehicle_condition                 0
Type_of_order                     0
Type_of_vehicle                   0
multiple_deliveries             993
Festival                        228
City                           1200
Time_taken(min)                   0
dtype: int64
